In [1]:
import pandas as pd
from sklearn.utils import resample
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

## Initial Data

In [2]:
initial_data = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')

In [3]:
initial_data

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21803,S025,MF2,27,MF2_FRQ6A,FRQ_Component,FRQ 6,KC.U1.09.quantitative_display_reading,KC.U1.09.quantitative_display_reading|KC.U7.11...,0.0,1,0,I,Rubric: identify the approximate numeric value...,I
21804,S025,MF2,27,MF2_FRQ6B,FRQ_Component,FRQ 6,KC.U7.12.mean_hypotheses,KC.U7.12.mean_hypotheses|KC.U7.11.matched_pair...,1.0,1,100,E,Rubric: define the population mean paired diff...,E
21805,S025,MF2,27,MF2_FRQ6C,FRQ_Component,FRQ 6,KC.U7.13.t_test_conditions,KC.U7.13.t_test_conditions|KC.U7.11.matched_pa...,0.5,1,50,P,"Rubric: address paired structure, random sampl...",P
21806,S025,MF2,27,MF2_FRQ6D,FRQ_Component,FRQ 6,KC.U7.14.t_test_statistic_mean,KC.U7.14.t_test_statistic_mean|KC.U7.15.p_valu...,1.0,1,100,E,"Rubric: use mean difference, standard error, d...",E


## Bootstrap samples

Create bootstrapped samples that maintains KC coverage :

In [4]:
def bootstrap_students(initial_data, target_students, seed = 42): 
    """Create bootstrap samples."""

     # Unique student list
    student_ids = initial_data["student_id"].unique()
    
    # Resample student IDs with replacement 
    sampled_ids = resample(
        student_ids,
        replace=True,
        n_samples=target_students,
        random_state=seed,
    )

    # Expand to full attempt rows with fresh IDs 
    new_rows = []
    for new_idx, orig_sid in enumerate(sampled_ids, start=1):
        all_attempts = initial_data[initial_data["student_id"] == orig_sid].copy()
        all_attempts["student_id"] = f"S{new_idx:03d}"
        new_rows.append(all_attempts)

    return pd.concat(new_rows, ignore_index=True)

In [5]:
sample_50 = bootstrap_students(initial_data, 50)
sample_100 = bootstrap_students(initial_data, 100)

## Diagnostics

In [ ]:
def print_diagnostics(original: pd.DataFrame, bootstrapped: pd.DataFrame, label: str):
    print(f"\n{'='*62}")
    print(f"  {label}")
    print(f"{'='*62}")
 
    n_orig = original["student_id"].nunique()
    n_boot = bootstrapped["student_id"].nunique()
    print(f"  Students  : {n_orig:>5}  →  {n_boot}")
    print(f"  Total rows: {len(original):>5}  →  {len(bootstrapped)}")
 
    # percent_score value counts (discrete: 0, 50, 100)
    print("\n  percent_score distribution:")
    orig_pct = original["percent_score"].value_counts(normalize=True).sort_index()
    boot_pct = bootstrapped["percent_score"].value_counts(normalize=True).sort_index()
    print(f"    {'value':<8} {'original':>10} {'bootstrapped':>14}")
    for v in sorted(set(orig_pct.index) | set(boot_pct.index)):
        print(f"    {str(v):<8} {orig_pct.get(v, 0):>10.3f} {boot_pct.get(v, 0):>14.3f}")

 
    # primary_kc_id distribution
    print("\n  primary_kc_id distribution:")
    orig_it = original["primary_kc_id"].value_counts(normalize=True).sort_index()
    boot_it = bootstrapped["primary_kc_id"].value_counts(normalize=True).sort_index()
    print(f"    {'type':<16} {'original':>10} {'bootstrapped':>14}")
    for itype in sorted(set(orig_it.index) | set(boot_it.index)):
        print(f"    {itype:<16} {orig_it.get(itype, 0):>10.3f} {boot_it.get(itype, 0):>14.3f}")
 
    # avg attempts per student
    orig_avg = len(original) / n_orig
    boot_avg = len(bootstrapped) / n_boot
    print(f"\n  Avg attempts/student: {orig_avg:.1f}  →  {boot_avg:.1f}")

In [24]:
def coverage_chart(data, col, label):
    prop_df = (
        data[col]
        .value_counts(normalize=True)
        .reset_index()
        .rename(columns={"index": col, "proportion": "proportion"})
    )

    chart = alt.Chart(prop_df).mark_bar().encode(
        x=alt.X(f'{col}:N', title='Assignment'),   # :N = nominal/categorical
        y=alt.Y('proportion:Q', title='Number of Attempts', stack="normalize"), # :Q = quantitative
        tooltip=[f'{col}:N', 'count():Q']
    ).properties(
        title=label,
        width=400,
        height=300
    )
    return chart

### Assignment Coverage

In [25]:
coverage_chart(initial_data, 'assignment_id', '25 Student Sample (Original data)') | coverage_chart(sample_50, 'assignment_id', '50 Student Sample') | coverage_chart(sample_100,'assignment_id', '100 Student Sample')

alt.HConcatChart(...)

### KC Coverage

In [ ]:
chart = alt.Chart(comparison).mark_bar().encode(
    x=alt.X('num_items', title='Number of items').bin(maxbins=15),
    y=alt.Y('count()', title='Number of KCs'),
    tooltip=['count()'] 
).facet(
    column = alt.Column('student_id', title = 'Student')
).properties(
    title='KC Coverage Imbalance'
)
chart

In [11]:
print_diagnostics(initial_data,sample_50, 'Number of Students : 50')


  Number of Students : 50
  Students  :    25  →  50
  Total rows: 21808  →  43581

  percent_score distribution:
    value      original   bootstrapped
    0             0.439          0.458
    50            0.065          0.064
    100           0.496          0.478

  Assignment coverage (proportion of rows):
    assignment     original   bootstrapped
    HW1               0.024          0.024
    HW10              0.036          0.036
    HW11              0.020          0.019
    HW12              0.019          0.019
    HW13              0.018          0.018
    HW14              0.017          0.017
    HW15              0.019          0.018
    HW16              0.020          0.019
    HW17              0.018          0.017
    HW18              0.019          0.018
    HW19              0.013          0.013
    HW2               0.028          0.028
    HW20              0.023          0.023
    HW21              0.018          0.017
    HW22              0.023          0.

In [12]:
print_diagnostics(initial_data,sample_100, 'Number of Students : 100')


  Number of Students : 100
  Students  :    25  →  100
  Total rows: 21808  →  87249

  percent_score distribution:
    value      original   bootstrapped
    0             0.439          0.448
    50            0.065          0.064
    100           0.496          0.488

  Assignment coverage (proportion of rows):
    assignment     original   bootstrapped
    HW1               0.024          0.024
    HW10              0.036          0.036
    HW11              0.020          0.019
    HW12              0.019          0.018
    HW13              0.018          0.018
    HW14              0.017          0.017
    HW15              0.019          0.018
    HW16              0.020          0.019
    HW17              0.018          0.017
    HW18              0.019          0.019
    HW19              0.013          0.014
    HW2               0.028          0.028
    HW20              0.023          0.024
    HW21              0.018          0.017
    HW22              0.023          